# Senologie-Kohortenanalyse (SQL on FHIR)

Dieses Notebook demonstriert die Auswertung der synthetischen Senologie-Kohorte (12 Patientinnen, 177 Ressourcen) über die sechs **SQL-on-FHIR v2 `ViewDefinition`**-Artefakte des IG (`input/fsh/views/*.json`).

## Ausführungsmodi

Das Notebook unterstützt drei austauschbare Ausführungsmodi für die ViewDefinitions. Der Modus wird in Zelle **`mode-selector`** (weiter unten) per Variable `EXECUTION_MODE` gewählt; alle folgenden Zellen arbeiten modusunabhängig identisch:

| Modus | `EXECUTION_MODE` | Engine | Voraussetzungen |
|---|---|---|---|
| **1 — Pathling Docker** *(empfohlen)* | `"docker"` | Pathling FHIR Server (`$run` auf `ViewDefinition`) | `docker compose -f docker-compose.pathling.yaml up -d` + `python3 scripts/load-to-pathling.py` |
| **2 — Pathling Python** | `"python"` | `pathling.PathlingContext` (lokales Spark) | `pip install pathling pandas` |
| **3 — Custom Evaluator** *(Fallback)* | `"custom"` | Im Notebook enthaltener Mini-FHIRPath-Runner | nur `pandas`, `requests` — kein Docker, kein Spark |

> [Pathling](https://pathling.csiro.au/) ist die quelloffene SQL-on-FHIR-Engine von CSIRO (Apache-2.0). Beide Pathling-Varianten führen die 6 ViewDefinitions identisch aus; Modus 3 bleibt als leichtgewichtiger Fallback erhalten und nutzt den HAPI-Testserver auf `http://localhost:8095/fhir` bzw. direkt die JSON-Ressourcen aus `fsh-generated/resources/`.

Schritt-für-Schritt-Anleitung zu den drei Modi: siehe `notebooks/README.md`.

## 0. Setup

In [ ]:
import json
import re
from pathlib import Path
from datetime import datetime, date
from typing import Any, Callable, Dict, Iterable, List, Optional

import pandas as pd
import requests
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='deep')
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

FHIR_BASE = 'http://localhost:8095/fhir'
VIEWS_DIR = Path('../input/fsh/views')
print(f'FHIR base: {FHIR_BASE}')
print(f'Views dir: {VIEWS_DIR.resolve()}')

### 0a. Ausführungsmodus wählen

`EXECUTION_MODE` bestimmt, welche Engine die ViewDefinitions ausführt. Bei Problemen mit Docker oder Pathling einfach auf `"custom"` setzen — die Analysezellen weiter unten sind identisch.

In [ ]:
# === Ausführungsmodus ========================================================
# "docker"  -> Pathling FHIR Server (via docker-compose.pathling.yaml)
# "python"  -> Pathling Python-Bibliothek (lokales Spark, benötigt `pip install pathling`)
# "custom"  -> Im Notebook enthaltener FHIRPath-Mini-Runner (Fallback, kein Pathling)
EXECUTION_MODE = "docker"

# Endpunkte / Pfade je nach Modus -- bei Bedarf hier anpassen.
PATHLING_URL      = "http://localhost:8090/fhir"   # Mode 1 (docker)
HAPI_URL          = "http://localhost:8095/fhir"   # Mode 3 (custom, HAPI-Fallback)
RESOURCES_DIR     = Path("../fsh-generated/resources")  # Mode 2 + Mode 3 (Datei-Fallback)

assert EXECUTION_MODE in {"docker", "python", "custom"}, EXECUTION_MODE
print(f"EXECUTION_MODE = {EXECUTION_MODE!r}")
if EXECUTION_MODE == "docker":
    print(f"  Pathling server: {PATHLING_URL}")
elif EXECUTION_MODE == "python":
    print(f"  Pathling Python-Lib, Bundles aus: {RESOURCES_DIR.resolve()}")
else:
    print(f"  Custom Runner, Ressourcen aus: {HAPI_URL} (Fallback auf {RESOURCES_DIR.resolve()})")

## 1. ViewDefinitions laden

In [ ]:
VIEWS: Dict[str, dict] = {}
for vd_file in sorted(VIEWS_DIR.glob('ViewDefinition-*.json')):
    vd = json.loads(vd_file.read_text())
    VIEWS[vd['name']] = vd
    cols = sum(len(s.get('column', [])) for s in vd.get('select', []))
    print(f"{vd['name']:28s}  resource={vd['resource']:18s}  columns={cols}")

## 2. FHIR-Ressourcen vom HAPI-Server laden

In [ ]:
# Ressourcen-Ladepolitik je Modus:
#   - "docker": Pathling hat die Ressourcen bereits per scripts/load-to-pathling.py
#     bekommen; hier wird nur die Erreichbarkeit geprüft.
#   - "python": die Pathling-Bibliothek liest die Bundles direkt aus RESOURCES_DIR.
#   - "custom": Ressourcen werden hier in einen In-Memory-Cache geladen, wahlweise
#     vom HAPI-Testserver oder direkt aus den FSH-generated JSON-Dateien.

RESOURCE_CACHE: Dict[str, List[dict]] = {}
required_types = sorted({vd['resource'] for vd in VIEWS.values()})

def _load_from_files(resources_dir: Path) -> Dict[str, List[dict]]:
    cache: Dict[str, List[dict]] = {rt: [] for rt in required_types}
    for p in sorted(resources_dir.glob('*.json')):
        try:
            res = json.loads(p.read_text())
        except Exception:
            continue
        rt = res.get('resourceType')
        if rt in cache:
            cache[rt].append(res)
    return cache

def _load_from_hapi(base: str) -> Dict[str, List[dict]]:
    cache: Dict[str, List[dict]] = {}
    for rt in required_types:
        url = f'{base}/{rt}?_count=200'
        acc: List[dict] = []
        while url:
            r = requests.get(url, headers={'Accept': 'application/fhir+json'}, timeout=30)
            r.raise_for_status()
            bundle = r.json()
            for entry in bundle.get('entry', []) or []:
                res = entry.get('resource')
                if res:
                    acc.append(res)
            url = next((l['url'] for l in bundle.get('link', []) if l.get('relation') == 'next'), None)
        cache[rt] = acc
    return cache

if EXECUTION_MODE == 'docker':
    r = requests.get(f'{PATHLING_URL}/metadata', headers={'Accept': 'application/fhir+json'}, timeout=10)
    r.raise_for_status()
    print(f'Pathling erreichbar: {PATHLING_URL} (CapabilityStatement OK)')
elif EXECUTION_MODE == 'python':
    if not RESOURCES_DIR.is_dir():
        raise FileNotFoundError(f'RESOURCES_DIR nicht gefunden: {RESOURCES_DIR.resolve()}')
    print(f'Pathling Python-Lib wird Bundles aus {RESOURCES_DIR.resolve()} lesen.')
else:  # custom
    try:
        RESOURCE_CACHE = _load_from_hapi(HAPI_URL)
        src = f'HAPI ({HAPI_URL})'
    except Exception as err:
        print(f'HAPI nicht erreichbar ({err}); fallback auf Dateien.')
        RESOURCE_CACHE = _load_from_files(RESOURCES_DIR)
        src = f'Dateien ({RESOURCES_DIR.resolve()})'
    for rt in required_types:
        print(f'{rt:20s}  {len(RESOURCE_CACHE.get(rt, [])):4d} Ressourcen  [{src}]')

## 3. Minimaler SQL-on-FHIR-Runner

Eine kleine FHIRPath-Engine ausreichend für die im IG definierten ViewDefinitions.

In [ ]:
# --- FHIRPath-Mini-Evaluator --------------------------------------------------
# Unterstützte Konstrukte:
#   feld.subfeld, feld[n], feld.where(expr), first(), exists(), count()
#   contains 'x', = 'x', != 'x', system = 'x' and code = 'y', or, and
#   startsWith('x'), contains('x'), matches('regex'), replaceMatches('re', 'sub')
#   ofType(TypeName), iif(cond, a, b), getResourceKey(), today(), empty(), toString()
#   string + string (Konkatenation), string | string (Fallback/Union), join('sep')
#
# Dies ist ein pragmatischer Runner, kein vollständiger FHIRPath-Parser.

Token = tuple  # (kind, value)

def tokenize(expr: str) -> List[Token]:
    tokens: List[Token] = []
    i, n = 0, len(expr)
    while i < n:
        c = expr[i]
        if c.isspace():
            i += 1
        elif c == "'":
            j = i + 1
            while j < n and expr[j] != "'":
                if expr[j] == '\\':
                    j += 2
                else:
                    j += 1
            tokens.append(('STR', expr[i+1:j].replace("\\'", "'")))
            i = j + 1
        elif c in '().,[]|+':
            tokens.append(('OP', c))
            i += 1
        elif c == '=' and (i + 1 >= n or expr[i+1] != '='):
            tokens.append(('OP', '='))
            i += 1
        elif expr[i:i+2] == '!=':
            tokens.append(('OP', '!='))
            i += 2
        elif c.isdigit():
            j = i
            while j < n and (expr[j].isdigit() or expr[j] == '.'):
                j += 1
            tokens.append(('NUM', expr[i:j]))
            i = j
        else:
            j = i
            while j < n and (expr[j].isalnum() or expr[j] in '_-:'):
                j += 1
            if j == i:
                i += 1
                continue
            word = expr[i:j]
            if word in ('and', 'or', 'contains', 'in', 'xor'):
                tokens.append(('OP', word))
            else:
                tokens.append(('ID', word))
            i = j
    return tokens

def to_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return [v for v in x if v is not None]
    return [x]

def truthy(x: Any) -> bool:
    if x is None:
        return False
    if isinstance(x, bool):
        return x
    if isinstance(x, list):
        return len(x) > 0 and any(truthy(v) for v in x)
    if isinstance(x, str):
        return len(x) > 0
    return True

def split_top_level(expr: str, sep: str) -> List[str]:
    depth = 0
    in_str = False
    out: List[str] = []
    buf: List[str] = []
    i = 0
    while i < len(expr):
        c = expr[i]
        if c == "'" and (i == 0 or expr[i-1] != '\\'):
            in_str = not in_str
            buf.append(c)
        elif not in_str and c == '(':
            depth += 1
            buf.append(c)
        elif not in_str and c == ')':
            depth -= 1
            buf.append(c)
        elif not in_str and depth == 0 and expr[i:i+len(sep)] == sep and (len(sep) > 1 or expr[i:i+len(sep)+1] != sep * 2):
            out.append(''.join(buf).strip())
            buf = []
            i += len(sep)
            continue
        else:
            buf.append(c)
        i += 1
    if buf:
        out.append(''.join(buf).strip())
    return [s for s in out if s]

def split_args(s: str) -> List[str]:
    return split_top_level(s, ',')

def eval_path(expr: str, ctx: Any, root: Any = None) -> Any:
    """Einfacher FHIRPath-Evaluator. ctx = aktueller Kontext, root = Wurzelressource."""
    expr = expr.strip()
    if root is None:
        root = ctx

    # Fallback | — nimm ersten nicht-leeren
    if '|' in expr:
        parts = split_top_level(expr, '|')
        if len(parts) > 1:
            vals = [eval_path(p, ctx, root) for p in parts]
            collected: List[Any] = []
            for v in vals:
                collected.extend(to_list(v))
            return collected if collected else None

    # Boolean or/and
    for op in (' or ', ' and '):
        parts = split_top_level(expr, op)
        if len(parts) > 1:
            if op.strip() == 'or':
                return any(truthy(eval_path(p, ctx, root)) for p in parts)
            return all(truthy(eval_path(p, ctx, root)) for p in parts)

    # String-Konkatenation mit '+'
    plus_parts = split_top_level(expr, '+')
    if len(plus_parts) > 1:
        strs: List[str] = []
        for p in plus_parts:
            v = eval_path(p, ctx, root)
            v = v[0] if isinstance(v, list) and v else v
            if v is None:
                return None
            strs.append(str(v))
        return ''.join(strs)

    # Vergleiche '=' '!='
    for op_sym in (' = ', ' != '):
        parts = split_top_level(expr, op_sym)
        if len(parts) == 2:
            lhs = eval_path(parts[0], ctx, root)
            rhs = eval_path(parts[1], ctx, root)
            lhs_v = lhs[0] if isinstance(lhs, list) and lhs else lhs
            rhs_v = rhs[0] if isinstance(rhs, list) and rhs else rhs
            eq = (str(lhs_v) == str(rhs_v)) if (lhs_v is not None and rhs_v is not None) else False
            return eq if op_sym.strip() == '=' else (not eq)

    # 'contains' als Operator (lhs contains 'str')
    contains_parts = split_top_level(expr, ' contains ')
    if len(contains_parts) == 2:
        lhs = eval_path(contains_parts[0], ctx, root)
        rhs = eval_path(contains_parts[1], ctx, root)
        rhs_v = rhs[0] if isinstance(rhs, list) and rhs else rhs
        if rhs_v is None:
            return False
        for v in to_list(lhs):
            if v is None:
                continue
            if isinstance(v, list):
                if rhs_v in v:
                    return True
            elif isinstance(v, str) and isinstance(rhs_v, str):
                if rhs_v in v:
                    return True
            elif v == rhs_v:
                return True
        return False

    # String literal
    if expr.startswith("'") and expr.endswith("'"):
        return expr[1:-1]

    # Numerisch
    if re.fullmatch(r'-?\d+(\.\d+)?', expr):
        return float(expr) if '.' in expr else int(expr)
    if expr in ('true', 'false'):
        return expr == 'true'
    if expr == '{}':
        return None

    # Funktion auf Root-Ebene?  iif(...), today(), getResourceKey()
    m = re.fullmatch(r'(\w+)\((.*)\)', expr, flags=re.DOTALL)
    if m and m.group(1) in ('iif', 'today', 'getResourceKey'):
        fn, arg = m.group(1), m.group(2)
        if fn == 'iif':
            args = split_args(arg)
            cond = truthy(eval_path(args[0], ctx, root))
            if cond:
                return eval_path(args[1], ctx, root) if len(args) > 1 else None
            return eval_path(args[2], ctx, root) if len(args) > 2 else None
        if fn == 'today':
            return date.today().isoformat()
        if fn == 'getResourceKey':
            return root.get('id') if isinstance(root, dict) else None

    # Pfad: tokenisieren und step-by-step auswerten
    steps = split_top_level(expr, '.')
    current: Any = ctx
    for step in steps:
        current = apply_step(step, current, root)
        if current is None:
            return None
    return current

def apply_step(step: str, ctx: Any, root: Any) -> Any:
    step = step.strip()
    # Funktionsaufruf foo(bar)?
    m = re.fullmatch(r'(\w+)\((.*)\)', step, flags=re.DOTALL)
    if m:
        fn, arg = m.group(1), m.group(2)
        items = to_list(ctx)
        if fn == 'where':
            return [it for it in items if truthy(eval_path(arg, it, root))]
        if fn == 'first':
            return items[0] if items else None
        if fn == 'last':
            return items[-1] if items else None
        if fn == 'count':
            return len(items)
        if fn == 'exists':
            return len(items) > 0
        if fn == 'empty':
            return len(items) == 0
        if fn == 'toString':
            return [str(it) for it in items if it is not None] or None
        if fn == 'startsWith':
            target = eval_path(arg, ctx, root)
            t = target[0] if isinstance(target, list) and target else target
            return any(isinstance(it, str) and isinstance(t, str) and it.startswith(t) for it in items)
        if fn == 'contains':
            target = eval_path(arg, ctx, root)
            t = target[0] if isinstance(target, list) and target else target
            return any(isinstance(it, str) and isinstance(t, str) and t in it for it in items)
        if fn == 'matches':
            target = eval_path(arg, ctx, root)
            t = target[0] if isinstance(target, list) and target else target
            if not isinstance(t, str):
                return False
            return any(isinstance(it, str) and re.search(t, it) is not None for it in items)
        if fn == 'replaceMatches':
            args = split_args(arg)
            pat = eval_path(args[0], ctx, root)
            sub = eval_path(args[1], ctx, root)
            pat_v = pat[0] if isinstance(pat, list) and pat else pat
            sub_v = sub[0] if isinstance(sub, list) and sub else sub
            out = [re.sub(pat_v, sub_v or '', it) for it in items if isinstance(it, str)]
            return out[0] if len(out) == 1 else (out or None)
        if fn == 'ofType':
            type_name = arg.strip()
            # In FHIR JSON sind getypte Felder z.B. value[x] bereits als valueDateTime etc. aufgelöst —
            # hier operieren wir auf dem Eltern-Objekt (z.B. extension), suchen also value<Type> oder deceased<Type> Kombis.
            prim_types = {'dateTime', 'date', 'string', 'boolean', 'decimal', 'integer', 'code', 'uri', 'instant'}
            complex_types = {'Period', 'Quantity', 'Reference', 'CodeableConcept', 'Coding', 'Range', 'Ratio'}
            out: List[Any] = []
            for it in items:
                if not isinstance(it, dict):
                    continue
                # wenn ctx direkt ein typisiertes Objekt ist, prüfe resourceType/type
                if type_name in prim_types or type_name in complex_types:
                    # suche Geschwisterschlüssel wie value<Type> – kann nur greifen wenn Aufruf auf value[x] gemachte wurde
                    # Da der Aufrufer uns meist via 'value' oder 'performed' usw. navigiert hat, haben wir schon den 
                    # aufgelösten Unterwert verloren. Stattdessen: schaue nach <prefix><Type> auf Root-Ebene.
                    # Pragmatisch: wenn it bereits ein dict mit 'start'/'end' (Period) oder 'value'/'unit' (Quantity) ist, passt es.
                    ok = True
                    if type_name == 'Period' and not ('start' in it or 'end' in it):
                        ok = False
                    if type_name == 'Quantity' and 'value' not in it:
                        ok = False
                    if ok:
                        out.append(it)
            if not out:
                return None
            return out[0] if len(out) == 1 else out
        if fn == 'join':
            sep = eval_path(arg, ctx, root)
            sep_v = sep[0] if isinstance(sep, list) and sep else sep
            strs = [str(it) for it in items if it is not None]
            return sep_v.join(strs) if strs else None
        # Unknown function
        return None
    # Indizierter Zugriff feld[n]?
    m = re.fullmatch(r'(\w+)\[(\d+)\]', step)
    if m:
        name, idx = m.group(1), int(m.group(2))
        inner = navigate(ctx, name)
        lst = to_list(inner)
        return lst[idx] if idx < len(lst) else None
    # Einfacher Feldzugriff
    return navigate(ctx, step)

def navigate(ctx: Any, name: str) -> Any:
    """Navigation in einen Feldnamen — behandelt Listen (flat-map) und choice-types (value[x])."""
    if ctx is None:
        return None
    if isinstance(ctx, list):
        out: List[Any] = []
        for item in ctx:
            v = navigate(item, name)
            if v is None:
                continue
            if isinstance(v, list):
                out.extend(v)
            else:
                out.append(v)
        return out if out else None
    if not isinstance(ctx, dict):
        return None
    if name in ctx:
        return ctx[name]
    # choice type: wenn 'value' gesucht, finde value<Type>
    for k in ctx:
        if k.startswith(name) and len(k) > len(name) and k[len(name)].isupper():
            return ctx[k]
    return None

# --- Runner -------------------------------------------------------------------
def run_view(view_def: dict, resources: List[dict]) -> pd.DataFrame:
    # where-Filter
    filtered = resources
    for w in view_def.get('where', []) or []:
        path = w['path']
        filtered = [r for r in filtered if truthy(eval_path(path, r, r))]
    # select: base columns + forEach groups
    rows: List[dict] = []
    selects = view_def.get('select', [])
    for res in filtered:
        base_row: Dict[str, Any] = {}
        foreach_rows: List[Dict[str, Any]] = []
        for sel in selects:
            if 'forEach' in sel:
                items = to_list(eval_path(sel['forEach'], res, res))
                for item in items:
                    fr: Dict[str, Any] = {}
                    for col in sel.get('column', []):
                        v = eval_path(col['path'], item, res)
                        if isinstance(v, list):
                            v = v[0] if v else None
                        fr[col['name']] = v
                    foreach_rows.append(fr)
            else:
                for col in sel.get('column', []):
                    v = eval_path(col['path'], res, res)
                    if isinstance(v, list):
                        v = v[0] if v else None
                    base_row[col['name']] = v
        if foreach_rows:
            for fr in foreach_rows:
                rows.append({**base_row, **fr})
        else:
            rows.append(base_row)
    return pd.DataFrame(rows)

print('Runner bereit.')

## 4. Views materialisieren (DataFrames)

In [ ]:
# View-Materialisierung je Modus.
# Alle drei Zweige produzieren dasselbe: ein Dict `df` mit pandas-DataFrames
# pro ViewDefinition-Name. Die Analysezellen weiter unten arbeiten darauf.

df: Dict[str, pd.DataFrame] = {}

def _run_view_pathling_server(vd: dict) -> pd.DataFrame:
    '''Mode 1: Pathling FHIR Server -- POST /ViewDefinition/$run.'''
    params = {
        'resourceType': 'Parameters',
        'parameter': [{'name': 'viewResource', 'resource': vd}],
    }
    r = requests.post(
        f'{PATHLING_URL}/ViewDefinition/$run',
        json=params,
        headers={
            'Content-Type': 'application/fhir+json',
            'Accept': 'application/json',
        },
        timeout=120,
    )
    r.raise_for_status()
    # $run response handling -- Pathling returns JSON rows or a Parameters wrapper.
    try:
        body = r.json()
    except ValueError:
        rows = [json.loads(line) for line in r.text.splitlines() if line.strip()]
        return pd.DataFrame(rows)
    if isinstance(body, list):
        return pd.DataFrame(body)
    if body.get('resourceType') == 'Parameters':
        for p in body.get('parameter', []) or []:
            if p.get('name') in ('result', 'rows'):
                if 'resource' in p and p['resource'].get('resourceType') == 'Bundle':
                    rows = [e.get('resource') or {} for e in p['resource'].get('entry', []) or []]
                    return pd.DataFrame(rows)
                if 'valueString' in p:
                    rows = [json.loads(line) for line in p['valueString'].splitlines() if line.strip()]
                    return pd.DataFrame(rows)
    if body.get('resourceType') == 'Bundle':
        rows = [e.get('resource') or {} for e in body.get('entry', []) or []]
        return pd.DataFrame(rows)
    return pd.DataFrame([body])

def _run_views_pathling_python() -> Dict[str, pd.DataFrame]:
    '''Mode 2: pathling.PathlingContext -- lokales Spark.'''
    from pathling import PathlingContext  # lazy import
    pc = PathlingContext.create()
    data = pc.read.bundles(
        str(RESOURCES_DIR.resolve()),
        resource_types=sorted({vd['resource'] for vd in VIEWS.values()}),
    )
    out: Dict[str, pd.DataFrame] = {}
    for name, vd in VIEWS.items():
        spark_df = pc.view.execute(vd, data)
        out[name] = spark_df.toPandas() if hasattr(spark_df, 'toPandas') else pd.DataFrame(spark_df)
    return out

if EXECUTION_MODE == 'docker':
    for name, vd in VIEWS.items():
        df[name] = _run_view_pathling_server(vd)
        print(f"{name:28s}  rows={len(df[name]):4d}  cols={len(df[name].columns)}  [pathling-docker]")
elif EXECUTION_MODE == 'python':
    df = _run_views_pathling_python()
    for name in VIEWS:
        print(f"{name:28s}  rows={len(df[name]):4d}  cols={len(df[name].columns)}  [pathling-python]")
else:  # custom
    for name, vd in VIEWS.items():
        resources = RESOURCE_CACHE.get(vd['resource'], [])
        df[name] = run_view(vd, resources)
        print(f"{name:28s}  rows={len(df[name]):4d}  cols={len(df[name].columns)}  [custom]")

# Alle Engines liefern `patientId` mit/ohne 'Patient/'-Präfix -- hier normalisieren,
# damit die Analysezellen weiter unten mit allen Modi funktionieren.
for key in ('DiagnoseKohorte', 'OperationenKohorte', 'PathologieKohorte', 'SystemtherapieKohorte', 'TumorboardKohorte'):
    if key in df and 'patientId' in df[key].columns:
        df[key]['patientId'] = df[key]['patientId'].astype(str).str.replace(r'^Patient/', '', regex=True)

In [ ]:
df['PatientKohorte'].head(20)

In [ ]:
df['DiagnoseKohorte'].head(20)

In [ ]:
df['OperationenKohorte'].head(20)

In [ ]:
df['PathologieKohorte'].head(20)

In [ ]:
df['SystemtherapieKohorte'].head(20)

In [ ]:
df['TumorboardKohorte'].head(20)

## 5. Ableitung: Alter, Subtyp, BET-Art

Ein paar heuristische Ableitungen, die auch ohne strukturierte IHC-Observations aus dem Freitext rekonstruiert werden:

In [ ]:
# Alter aus birthDate berechnen
pat = df['PatientKohorte'].copy()
pat['birthDate'] = pd.to_datetime(pat['birthDate'], errors='coerce')
today_dt = pd.Timestamp(date.today())
pat['age'] = ((today_dt - pat['birthDate']).dt.days / 365.25).round(1)
df['PatientKohorte'] = pat

# Subtyp aus Pathologie + Diagnose ableiten
patho = df['PathologieKohorte'].copy()
patho['datum'] = pd.to_datetime(patho['datum'], errors='coerce')

diag = df['DiagnoseKohorte'].copy()
# DCIS-Markierung
diag['isDCIS'] = diag['snomed'].astype(str).isin(['109889007', '254838004']) | diag['icd10'].astype(str).str.startswith('D05')

# Subtyp pro Patient: aus dem _letzten_ Pathologiebefund pro Patient
patho_sorted = patho.sort_values(['patientId', 'datum']).groupby('patientId').tail(1)

def subtype_for(row) -> str:
    er, pr, her2 = row.get('er'), row.get('pr'), row.get('her2')
    if pd.isna(er) and pd.isna(pr) and pd.isna(her2):
        return 'unbekannt'
    if her2 == 'positive':
        return 'HER2+'
    if (er == 'negative' and pr == 'negative' and her2 == 'negative'):
        return 'TNBC'
    if (er == 'positive' or pr == 'positive') and (her2 in ('negative', None, float('nan'))):
        return 'HR+/HER2-'
    return 'unklar'

patho_sorted['subtyp'] = patho_sorted.apply(subtype_for, axis=1)

# DCIS überschreibt (reine DCIS-Fälle gehen vor)
dcis_patients = set(diag[diag['isDCIS']]['patientId'].unique()) - set(
    diag[(~diag['isDCIS']) & (diag['snomed'] == '254837009')]['patientId'].unique()
)
pat_subtype = patho_sorted.set_index('patientId')['subtyp']
for pid in dcis_patients:
    pat_subtype[pid] = 'DCIS'

# Patienten ohne Patho aber mit DCIS-Diagnose
for pid in dcis_patients:
    if pid not in pat_subtype.index:
        pat_subtype[pid] = 'DCIS'

df['PatientKohorte']['subtyp'] = df['PatientKohorte']['id'].map(pat_subtype).fillna('unbekannt')
df['PatientKohorte'][['id', 'name', 'age', 'subtyp']].head(15)

## Analyse 1 — Fallzahl pro Subtyp

Verteilung der Patientinnen auf die vier Haupt-Subtypen (HR+/HER2-, HER2+, TNBC, DCIS).

In [ ]:
subtype_counts = df['PatientKohorte']['subtyp'].value_counts()
print(subtype_counts)

fig, ax = plt.subplots(figsize=(7, 4))
subtype_counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato', 'goldenrod', 'mediumseagreen', 'slategray', 'lightgray'])
ax.set_title('Fallzahl pro Subtyp')
ax.set_xlabel('Subtyp')
ax.set_ylabel('Patientinnen')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Analyse 2 — BET-Rate

Anteil brusterhaltender Operationen an allen Brust-OPs (BET vs. Mastektomie). DKG-Qualitätsindikator.

In [ ]:
op = df['OperationenKohorte'].copy()
brust_ops = op[op['art'].isin(['BET', 'Mastektomie'])]
print(f'Brust-Operationen gesamt: {len(brust_ops)}')

bet_count = int((brust_ops['art'] == 'BET').sum())
mast_count = int((brust_ops['art'] == 'Mastektomie').sum())
bet_rate = bet_count / max(1, bet_count + mast_count) * 100
print(f'BET: {bet_count}, Mastektomie: {mast_count}')
print(f'BET-Rate: {bet_rate:.1f}%')

fig, ax = plt.subplots(figsize=(5, 4))
brust_ops['art'].value_counts().plot(kind='bar', ax=ax, color=['seagreen', 'coral'])
ax.set_title(f'BET-Rate: {bet_rate:.1f}%')
ax.set_xlabel('OP-Art')
ax.set_ylabel('Anzahl')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Analyse 3 — R0-Rate

Anteil R0-Resektionen an allen dokumentierten Brust-OPs. Qualitätsindikator QI-11 (Ziel: möglichst hoch).

In [ ]:
op_with_r = op[op['outcome'].notna()]
print(f'OPs mit R-Status dokumentiert: {len(op_with_r)} von {len(op)} gesamt')

r_counts = op_with_r['outcome'].value_counts()
print(r_counts)

r0_rate = (op_with_r['outcome'] == 'R0').sum() / max(1, len(op_with_r)) * 100
print(f'R0-Rate: {r0_rate:.1f}%')

fig, ax = plt.subplots(figsize=(5, 4))
r_counts.plot(kind='bar', ax=ax, color=['mediumseagreen', 'orange', 'firebrick'])
ax.set_title(f'R-Status Verteilung (R0-Rate: {r0_rate:.1f}%)')
ax.set_xlabel('R-Status')
ax.set_ylabel('Anzahl')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Analyse 4 — Altersverteilung

Boxplot der Patientinnenalter, optional stratifiziert nach Subtyp.

In [ ]:
pat_age = df['PatientKohorte'].dropna(subset=['age']).copy()
print(pat_age[['age']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(pat_age['age'], vert=True, patch_artist=True,
                 boxprops=dict(facecolor='lightsteelblue'))
axes[0].set_title('Altersverteilung (alle)')
axes[0].set_ylabel('Alter in Jahren')
axes[0].set_xticklabels(['Gesamt'])

if HAS_SNS:
    sns.boxplot(data=pat_age, x='subtyp', y='age', ax=axes[1])
else:
    groups = pat_age.groupby('subtyp')['age'].apply(list)
    axes[1].boxplot(groups.values, labels=groups.index)
axes[1].set_title('Alter nach Subtyp')
axes[1].set_xlabel('Subtyp')
axes[1].set_ylabel('Alter in Jahren')

plt.tight_layout()
plt.show()

## Analyse 5 — Therapie-Mix pro Subtyp

Welche Systemtherapie-Art (CH = Chemo, HO = Hormon, IM = Immun, ZT = Zielgerichtet) pro Subtyp dokumentiert ist.

In [ ]:
sys = df['SystemtherapieKohorte'].copy()
sys = sys.merge(df['PatientKohorte'][['id', 'subtyp']].rename(columns={'id': 'patientId'}),
                on='patientId', how='left')

therapy_mix = pd.crosstab(sys['subtyp'], sys['art']).fillna(0).astype(int)
print(therapy_mix)

fig, ax = plt.subplots(figsize=(8, 4))
therapy_mix.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_title('Therapie-Mix pro Subtyp')
ax.set_xlabel('Subtyp')
ax.set_ylabel('Anzahl Therapien')
ax.legend(title='Therapie', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Analyse 6 — Time-to-Treatment

Verteilung der Zeitspanne von Diagnose bis zur Erst-Operation (in Tagen). Relevanter Indikator der Versorgungsqualität.

In [ ]:
diag2 = df['DiagnoseKohorte'].copy()
diag2['diagnosedatum'] = pd.to_datetime(diag2['diagnosedatum'], errors='coerce')
diag2['onsetDateTime'] = pd.to_datetime(diag2['onsetDateTime'], errors='coerce')
diag2['recordedDate'] = pd.to_datetime(diag2['recordedDate'], errors='coerce')
diag2['diag_datum'] = diag2['diagnosedatum'].fillna(diag2['onsetDateTime']).fillna(diag2['recordedDate'])

first_diag = diag2.sort_values(['patientId', 'diag_datum']).groupby('patientId')['diag_datum'].first()

op2 = df['OperationenKohorte'].copy()
op2['datum'] = pd.to_datetime(op2['datum'].fillna(op2['beginn']), errors='coerce')
first_op = op2[op2['art'].isin(['BET', 'Mastektomie'])].sort_values(['patientId', 'datum']).groupby('patientId')['datum'].first()

tt = pd.concat({'diagnose': first_diag, 'op': first_op}, axis=1).dropna()
tt['tage'] = (tt['op'] - tt['diagnose']).dt.days
tt = tt[tt['tage'] >= 0]
print(tt[['tage']].describe().round(1))
print(f'\nMedian Time-to-Treatment: {tt["tage"].median():.0f} Tage')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tt['tage'], bins=12, color='steelblue', edgecolor='white')
ax.axvline(tt['tage'].median(), color='red', linestyle='--', label=f'Median {tt["tage"].median():.0f}d')
ax.axvline(28, color='gray', linestyle=':', label='Richtwert 28 Tage')
ax.set_title('Time-to-Treatment: Diagnose → Erst-OP')
ax.set_xlabel('Tage')
ax.set_ylabel('Anzahl Patientinnen')
ax.legend()
plt.tight_layout()
plt.show()

## Analyse 7 — Crosstab Subtyp × OP-Art

Welcher Subtyp erhält welche OP? Erwartet: DCIS → vermehrt BET, HER2+/TNBC → häufiger Mastektomie in höheren Stadien.

In [ ]:
op3 = df['OperationenKohorte'].copy()
op3 = op3.merge(df['PatientKohorte'][['id', 'subtyp']].rename(columns={'id': 'patientId'}),
                on='patientId', how='left')
op3 = op3[op3['art'].isin(['BET', 'Mastektomie', 'SNB', 'ALND'])]

ct = pd.crosstab(op3['subtyp'], op3['art']).fillna(0).astype(int)
print(ct)

fig, ax = plt.subplots(figsize=(8, 4))
if HAS_SNS:
    sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', ax=ax, cbar=False)
else:
    im = ax.imshow(ct.values, cmap='YlGnBu', aspect='auto')
    for (i, j), v in pd.DataFrame(ct).stack().items():
        ax.text(list(ct.columns).index(j), list(ct.index).index(i), str(v), ha='center', va='center')
    ax.set_xticks(range(len(ct.columns)), ct.columns)
    ax.set_yticks(range(len(ct.index)), ct.index)
ax.set_title('OP-Art × Subtyp')
plt.tight_layout()
plt.show()

## Zusammenfassung

Die Analyse zeigt exemplarisch, wie die SQL-on-FHIR-Views als Grundlage für:

- Qualitätsindikatoren (BET-Rate, R0-Rate, Time-to-Treatment)
- Subtypen-Stratifizierung (HR+/HER2-, HER2+, TNBC, DCIS)
- Therapie-Mix-Analysen
- Versorgungspfad-Analysen

verwendet werden können. Für produktive Auswertungen:

1. Die Heuristiken zur Subtyp-/R-Status-Ableitung durch strukturierte Observations (ER%, PR%, HER2-Score, Ki-67%, R-Status als separate Observation oder Bundesland-Kodierung) ersetzen.
2. Die Views mit einem produktionsreifen Runner ([Pathling](https://pathling.csiro.au/), [sof-js](https://github.com/FHIR/sql-on-fhir-v2/tree/master/sof-js)) ausführen.
3. Ergebnis-Tables über eine BI-Plattform (Superset, Metabase, Tableau) visualisieren oder als Parquet/Delta-Datei im Data Lake persistieren.
4. Pseudonymisierung und Governance gemäß MII-Broad Consent sicherstellen.